# IKG SQL Metadata Extractor

This companion notebook wraps the Python driver in `ikg_metadata_extractor.py` so you can run the full metadata harvesting workflow (GitLab download → SQL parsing → Excel export → Greenplum load) interactively.

## 1. Configure runtime variables
Fill in (or override) the environment variables below. Secrets such as the GitLab private token and Greenplum password are requested interactively when missing so they never appear in plain text inside the notebook.

In [ ]:
import getpass
import os
from pathlib import Path

DEFAULTS = {
    "GITLAB_URL": "https://devcloud.ubs.net",
    "PROJECT_ID": "ubs/gwma/smart-technology-and-analytics/staat-data-science/staat-ds-genesis/genesis-platform/ikg-dags",
    "BRANCH": "ikg-master",
    "SQL_FOLDER": "dags/ikg/scripts/sql",
    "EXCLUDE_FOLDER": "ikg_new fa_shhp_map",
    "OUTPUT_FILE": "ikg_metadata_<date>_<timestamp>.xls",
    "GREENPLUM_HOST": "greenplum-rdsp.zur.swissbank.com",
    "GREENPLUM_PORT": "5432",
    "GREENPLUM_DB": "gprdsp",
    "GREENPLUM_USER": "ds_rdsp_dev",
    "GREENPLUM_SCHEMA": "core_wma_shared,core_model,core_ikg,core_etl",
    "GREENPLUM_TARGET_TABLE": "sandbox_prj_smart_insights.ikg_metadata_auto_refresh",
    "GREENPLUM_TABLE_OWNER": "erd_gpdbprj_smart_insights",
    "GREENPLUM_TABLE_READER_ROLE": "erd_gpdb_prj_smart_insights_ro",
    "LOG_LEVEL": "DEBUG",
}

for key, value in DEFAULTS.items():
    os.environ.setdefault(key, value)

os.environ.setdefault("OUTPUT_DIR", str(Path.cwd()))

if not os.environ.get("PRIVATE_TOKEN"):
    os.environ["PRIVATE_TOKEN"] = getpass.getpass("GitLab Private Token: ")

if not os.environ.get("GREENPLUM_PASSWORD"):
    os.environ["GREENPLUM_PASSWORD"] = getpass.getpass("Greenplum Password: ")

print("Configuration complete. Excel will be written to:", Path(os.environ["OUTPUT_DIR"]) / os.environ["OUTPUT_FILE"])

## 2. Run the metadata pipeline
This cell invokes the reusable Python module to download SQL, parse lineage with sqlglot, publish the Excel workbook, and refresh the Greenplum metadata table.

In [ ]:
from ikg_metadata_extractor import Settings, run_pipeline

settings = Settings()
output_path = run_pipeline(settings)
output_path